# QAOA for MaxCut via uniqx uniqx

**Quantum Approximate Optimization Algorithm on real hardware — CPU, GPU, or QPU**

The [MaxCut problem](https://en.wikipedia.org/wiki/Maximum_cut) asks: given a graph $G = (V, E)$,
find a partition of the vertices into two sets that maximises the number of edges crossing the cut.
MaxCut is NP-hard in general, but QAOA provides a quantum-classical hybrid heuristic that maps
naturally onto quantum hardware.

## QAOA structure

QAOA alternates two unitaries, each parameterised by a variational angle:

$$|\boldsymbol{\gamma}, \boldsymbol{\beta}\rangle = \prod_{l=1}^{p} e^{-i \beta_l B}\, e^{-i \gamma_l C} |+\rangle^{\otimes n}$$

where:

- **Cost unitary** $e^{-i \gamma C}$: encodes the graph via the cost Hamiltonian
 $$C = \sum_{(i,j) \in E} \tfrac{1}{2}(I - Z_i Z_j)$$
 Each edge contributes $+1$ when qubits $i$ and $j$ are in different computational-basis states.

- **Mixer unitary** $e^{-i \beta B}$: drives exploration via the transverse-field mixer
 $$B = \sum_{i=1}^{n} X_i$$

Both are **matrix exponentials** (`ops.expv`) — uniqx's planner routes these to
Trotterised circuits on a QPU, dense exponentiation on a GPU, or classical simulation on a CPU,
depending on system size and available hardware.

The cost function $\langle C \rangle = \langle \boldsymbol{\gamma}, \boldsymbol{\beta} | C | \boldsymbol{\gamma}, \boldsymbol{\beta} \rangle$
is an **expectation value** (`ops.expect`) — on a QPU this becomes Pauli-$Z$ measurements;
on classical hardware it is a matrix-vector product.

We sweep $(\gamma, \beta)$ to find the angles that maximise the expected cut value,
then compare against the exact (brute-force) solution.

## 1. Setup

In [ ]:
import os
from uniqx import connect, login

endpoint = os.environ.get("UNIQX_GATEWAY", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=endpoint)
client = connect(endpoint)
print("Connected to", endpoint)
import math
import time
import numpy as np
import matplotlib.pyplot as plt

from uniqx import ops, tracing, fmt_vec, fmt_mat, fmt_scalar, parse_result
from uniqx.ops import SX, SZ, I2
from uniqx.core.execution import connect, preflight, submit, get

API_KEY = os.environ.get("UNIQX_API_KEY")


## 2. Graph Definition

We use a 5-node graph with 6 edges — rich enough to have a non-trivial MaxCut,
small enough to sweep the full parameter landscape and verify against brute force.

```
 0 --- 1
 |\ /|
 | \ / |
 | 3 |
 | / \ |
 |/ \|
 2 --- 4
```

In [ ]:
N_QUBITS = 5
DIM = 2**N_QUBITS  # Hilbert-space dimension

# Edges as (i, j) pairs
edges = [
    (0, 1),
    (0, 2),
    (0, 3),
    (1, 3),
    (1, 4),
    (2, 4),
]

print(f"Graph: {N_QUBITS} vertices, {len(edges)} edges")
print(f"Hilbert space dimension: {DIM}")
print(f"Edges: {edges}")

# --- Visualise the graph ---
fig, ax = plt.subplots(1, 1, figsize=(5, 5))

# Layout: place vertices on a circle
angles = [2 * math.pi * k / N_QUBITS - math.pi / 2 for k in range(N_QUBITS)]
pos = {k: (math.cos(a), math.sin(a)) for k, a in enumerate(angles)}

# Draw edges
for i, j in edges:
    xi, yi = pos[i]
    xj, yj = pos[j]
    ax.plot([xi, xj], [yi, yj], "k-", linewidth=1.5, zorder=1)

# Draw vertices
for k, (x, y) in pos.items():
    ax.scatter(x, y, s=600, c="#2563eb", edgecolors="black", linewidths=2, zorder=2)
    ax.text(
        x,
        y,
        str(k),
        ha="center",
        va="center",
        fontsize=14,
        fontweight="bold",
        color="white",
        zorder=3,
    )

ax.set_aspect("equal")
ax.set_title(f"MaxCut graph: {N_QUBITS} vertices, {len(edges)} edges", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Build Hamiltonians

### Cost Hamiltonian

$$C = \sum_{(i,j) \in E} \tfrac{1}{2}\bigl(I - Z_i Z_j\bigr)$$

The eigenvalues of $C$ are exactly the cut values for each computational-basis state.

### Mixer Hamiltonian

$$B = \sum_{i=0}^{n-1} X_i$$

We build both as dense $2^n \times 2^n$ matrices using pure-Python helpers
(matching the pattern used across uniqx example notebooks).

In [ ]:
# --- Pure-Python matrix helpers (no numpy, for tracing compatibility) ---


def kron(A, B):
    """Kronecker product of two 2D lists."""
    rA, rB = len(A), len(B)
    n = rA * rB
    C = [[0.0] * n for _ in range(n)]
    for i in range(rA):
        for j in range(rA):
            for k in range(rB):
                for l in range(rB):
                    C[i * rB + k][j * rB + l] = A[i][j] * B[k][l]
    return C


def eye(n):
    """n x n identity as 2D list."""
    return [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]


def matadd(A, B):
    """Element-wise addition of two 2D lists."""
    return [[A[i][j] + B[i][j] for j in range(len(A))] for i in range(len(A))]


def matscale(s, A):
    """Scalar multiplication of a 2D list."""
    return [[s * x for x in row] for row in A]


def embed_local(op, qubit, n_qubits):
    """Embed a single-qubit op into n-qubit Hilbert space."""
    result = eye(1)
    for k in range(n_qubits):
        result = kron(result, op if k == qubit else I2)
    return result


def two_body_local(opA, opB, qa, qb, n_qubits):
    """Two-qubit interaction: opA on qa, opB on qb, identity elsewhere."""
    parts = []
    for k in range(n_qubits):
        if k == qa:
            parts.append(opA)
        elif k == qb:
            parts.append(opB)
        else:
            parts.append(I2)
    result = eye(1)
    for p in parts:
        result = kron(result, p)
    return result


def build_cost_hamiltonian(edges, n_qubits):
    """C = sum_{(i,j)} 0.5 * (I - Z_i Z_j).

    Note: SZ = sigma_z / 2, so Z_i Z_j = 4 * SZ_i SZ_j.
    Therefore 0.5*(I - Z_i Z_j) = 0.5*I - 2*SZ_i*SZ_j.
    """
    dim = 2**n_qubits
    C = [[0.0] * dim for _ in range(dim)]
    I_full = eye(dim)
    for i, j in edges:
        ZZ = two_body_local(SZ, SZ, i, j, n_qubits)
        term = matadd(matscale(0.5, I_full), matscale(-2.0, ZZ))
        C = matadd(C, term)
    return C


def build_mixer_hamiltonian(n_qubits):
    """B = sum_i X_i = sum_i 2*SX_i (since SX = sigma_x / 2)."""
    dim = 2**n_qubits
    B = [[0.0] * dim for _ in range(dim)]
    for i in range(n_qubits):
        B = matadd(B, matscale(2.0, embed_local(SX, i, n_qubits)))
    return B


# Build the Hamiltonians
C_ham = build_cost_hamiltonian(edges, N_QUBITS)
B_ham = build_mixer_hamiltonian(N_QUBITS)

# Verify: C should be diagonal in the computational basis
diag_C = [C_ham[i][i] for i in range(DIM)]
off_diag_norm = sum(abs(C_ham[i][j]) for i in range(DIM) for j in range(DIM) if i != j)
print(f"Cost Hamiltonian: {DIM}x{DIM}")
print(f"  Diagonal (first 8): {[f'{d:.1f}' for d in diag_C[:8]]}")
print(f"  Off-diagonal norm:  {off_diag_norm:.6f} (should be 0)")
print(f"  Max cut value:      {max(diag_C):.1f} (= max eigenvalue of C)")
print(f"\nMixer Hamiltonian: {DIM}x{DIM} (off-diagonal)")

## 4. Brute-Force MaxCut

For our small graph we can enumerate all $2^n$ partitions and find the exact maximum cut.
This gives us the ground truth to compare QAOA against.

In [ ]:
def brute_force_maxcut(edges, n_vertices):
    """Enumerate all 2^n partitions, return (max_cut, best_bitstrings)."""
    best_cut = 0
    best_partitions = []
    for bits in range(2**n_vertices):
        cut = 0
        for i, j in edges:
            bi = (bits >> i) & 1
            bj = (bits >> j) & 1
            if bi != bj:
                cut += 1
        if cut > best_cut:
            best_cut = cut
            best_partitions = [bits]
        elif cut == best_cut:
            best_partitions.append(bits)
    return best_cut, best_partitions


exact_maxcut, exact_partitions = brute_force_maxcut(edges, N_QUBITS)

print(f"Exact MaxCut value: {exact_maxcut} (out of {len(edges)} edges)")
print(f"Number of optimal partitions: {len(exact_partitions)}")
print("\nOptimal partitions (vertex -> set):")
for bits in exact_partitions:
    partition = [str((bits >> k) & 1) for k in range(N_QUBITS)]
    set0 = [k for k in range(N_QUBITS) if (bits >> k) & 1 == 0]
    set1 = [k for k in range(N_QUBITS) if (bits >> k) & 1 == 1]
    print(f"  |{''.join(partition)}> : S0={set0}, S1={set1}")

# Compute all cut values for later comparison
all_cuts = []
for bits in range(2**N_QUBITS):
    cut = sum(1 for (i, j) in edges if ((bits >> i) & 1) != ((bits >> j) & 1))
    all_cuts.append(cut)

print(
    f"\nCut value distribution: {dict(sorted(set((v, all_cuts.count(v)) for v in all_cuts)))}"
)

## 5. QAOA Circuit as Traced Primitives (p=1)

A single QAOA layer applies the cost unitary $e^{-i\gamma C}$ followed by the
mixer unitary $e^{-i\beta B}$ to the initial state $|+\rangle^{\otimes n}$.

Both exponentials are traced as `ops.expv` calls, which uniqx compiles
to the best available backend (Trotter on QPU, dense expm on GPU/CPU).
The expectation value `ops.expect(C, psi)` gives the expected cut value.

In [ ]:
# Initial state: |+>^n = H^n |0>^n = uniform superposition
psi_plus = [1.0 / math.sqrt(DIM)] * DIM


@tracing.to_module(name="qaoa_p1")
def qaoa_p1(C_ham, B_ham, psi, gamma, beta):
    """Single-layer QAOA: cost unitary -> mixer unitary -> measure cost."""
    # Cost unitary: exp(-i * gamma * C)
    psi1 = ops.expv(C_ham, psi, gamma, hermitian=True)
    # Mixer unitary: exp(-i * beta * B)
    psi2 = ops.expv(B_ham, psi1, beta, hermitian=True)
    # Expectation value of the cost Hamiltonian
    cost = ops.expect(C_ham, psi2)
    return psi2, cost


# Trace with dummy parameters to build the module
mod_p1 = qaoa_p1(C_ham, B_ham, psi_plus, 0.5, 0.5)

ir_text = mod_p1.to_text()
n_ops = len(mod_p1.functions[0].ops)
n_params = len(mod_p1.functions[0].params)
n_results = len(mod_p1.functions[0].results)

print(f"QAOA p=1 module: {n_ops} ops, {n_params} params, {n_results} results")
print(f"IR size: {len(ir_text)} chars")
print(f"\nFirst 500 chars of IR:\n{ir_text[:500]}...")

## 6. Preflight Analysis

Before executing, uniqx's planner analyses the computation graph
and scores it on all available hardware backends. This reveals:

- **Time**: estimated wall-clock per execution
- **Cost**: USD per execution (cloud pricing model)
- **Error**: maximum error rate (QPU noise, numerical precision)
- **Carbon**: estimated CO$_2$ footprint

For QAOA, the `expv` nodes are the bottleneck — on a QPU they become
native Trotter circuits; on a GPU they are dense matrix exponentials.

In [ ]:
options = preflight(mod_p1, client=client)

print(f"Preflight for QAOA p=1 ({N_QUBITS} qubits, dim={DIM})")
print(f"Job ID: {options.job_id}\n")

hdr = f"  {'Label':<24} {'Time':>10} {'Cost ($)':>12} {'Error':>8} {'Carbon (g)':>11}"
print(hdr)
print(f"  {'-' * 24} {'-' * 10} {'-' * 12} {'-' * 8} {'-' * 11}")
for opt in options:
    flag = "  <-- rec" if opt.get("recommended") else ""
    print(
        f"  {opt['label']:<24} {opt['total_time']:>10.2f}"
        f"  ${opt['total_cost_usd']:>10.6f}"
        f"  {opt['max_error_rate']:>8.4f}"
        f"  {opt['total_carbon_g']:>10.3f}{flag}"
    )

SELECTED_OPTION = options.recommended["_idx"] if options.recommended else 0
print(f"\nSelected option: {SELECTED_OPTION}")

## 7. Parameter Sweep: Energy Landscape

QAOA p=1 has two variational parameters:
- $\gamma \in [0, 2\pi]$ — controls the cost unitary phase
- $\beta \in [0, \pi]$ — controls the mixer rotation angle

We sweep both on a grid, submit each $(\gamma, \beta)$ pair to uniqx,
and collect $\langle C \rangle$ to map out the 2D energy landscape.

The optimal angles maximise $\langle C \rangle$ (= expected cut value).

In [ ]:
N_GAMMA = 4
N_BETA = 4

gamma_vals = np.linspace(0.05, 2 * np.pi, N_GAMMA)
beta_vals = np.linspace(0.05, np.pi, N_BETA)

cost_landscape = np.zeros((N_GAMMA, N_BETA))

print(f"Sweeping {N_GAMMA} x {N_BETA} = {N_GAMMA * N_BETA} grid points...")
t0_sweep = time.monotonic()

for ig, gamma in enumerate(gamma_vals):
    for ib, beta in enumerate(beta_vals):
        mod = qaoa_p1(C_ham, B_ham, psi_plus, float(gamma), float(beta))

        jid = submit(
            mod,
            client=client,
            runtime_inputs=[
                fmt_mat(C_ham, DIM, DIM),
                fmt_mat(B_ham, DIM, DIM),
                fmt_vec(psi_plus, DIM),
                fmt_scalar(float(gamma)),
                fmt_scalar(float(beta)),
            ],
            preflight_job_id=options.job_id,
            option_idx=SELECTED_OPTION,
        )
        res = get(jid, client=client)

        payload = res.get("payload", b"")
        if isinstance(payload, str):
            payload = payload.encode()
        out = parse_result(payload, ["psi_out", "cost"])
        cost_landscape[ig, ib] = out["cost"][2][0]

    elapsed = time.monotonic() - t0_sweep
    print(f"  gamma {ig + 1}/{N_GAMMA} done ({elapsed:.1f}s)")

t_sweep = time.monotonic() - t0_sweep
best_idx = np.unravel_index(cost_landscape.argmax(), cost_landscape.shape)
print(f"\nSweep complete: {N_GAMMA * N_BETA} jobs in {t_sweep:.1f}s")
print(
    f"Max <C> = {cost_landscape.max():.4f} at"
    f" (gamma={gamma_vals[best_idx[0]]:.3f},"
    f" beta={beta_vals[best_idx[1]]:.3f})"
)

In [ ]:
# --- Plot the 2D energy landscape ---
fig, ax = plt.subplots(1, 1, figsize=(9, 6))

G_grid, B_grid = np.meshgrid(gamma_vals, beta_vals, indexing="ij")
im = ax.pcolormesh(G_grid, B_grid, cost_landscape, cmap="RdYlBu_r", shading="auto")
cbar = fig.colorbar(im, ax=ax, label=r"$\langle C \rangle$ (expected cut)")

# Mark the optimum
ax.plot(
    gamma_vals[best_idx[0]],
    beta_vals[best_idx[1]],
    "k*",
    markersize=18,
    label="best found",
)

ax.set_xlabel(r"$\gamma$", fontsize=13)
ax.set_ylabel(r"$\beta$", fontsize=13)
ax.set_title(f"QAOA p=1 Energy Landscape ({N_QUBITS}-vertex MaxCut)", fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 8. Optimal Solution (p=1)

Extract the best $(\gamma^*, \beta^*)$ from the sweep, re-run to get the
output state, and compare the expected cut value to the exact MaxCut.

In [ ]:
# Best parameters from the sweep
best_gamma_p1 = float(gamma_vals[best_idx[0]])
best_beta_p1 = float(beta_vals[best_idx[1]])
best_cost_p1 = cost_landscape[best_idx]

print("QAOA p=1 optimal parameters:")
print(f"  gamma* = {best_gamma_p1:.4f}")
print(f"  beta*  = {best_beta_p1:.4f}")
print(f"  <C>    = {best_cost_p1:.4f}")
print(f"\nExact MaxCut = {exact_maxcut}")
print(f"Approximation ratio: {best_cost_p1 / exact_maxcut:.4f}")

# Re-run at optimal angles to get the output state vector
mod_opt = qaoa_p1(C_ham, B_ham, psi_plus, best_gamma_p1, best_beta_p1)

jid = submit(
    mod_opt,
    client=client,
    runtime_inputs=[
        fmt_mat(C_ham, DIM, DIM),
        fmt_mat(B_ham, DIM, DIM),
        fmt_vec(psi_plus, DIM),
        fmt_scalar(best_gamma_p1),
        fmt_scalar(best_beta_p1),
    ],
    preflight_job_id=options.job_id,
    option_idx=SELECTED_OPTION,
)
res = get(jid, client=client)

payload = res.get("payload", b"")
if isinstance(payload, str):
    payload = payload.encode()
out = parse_result(payload, ["psi_out", "cost"])

# The output state is complex: [psi_re; psi_im] of length 2*DIM
psi_out = np.array(out["psi_out"][2])
if len(psi_out) == 2 * DIM:
    psi_re = psi_out[:DIM]
    psi_im = psi_out[DIM:]
    probs = psi_re**2 + psi_im**2
else:
    probs = psi_out**2

# Show probability distribution over computational basis states
print("\nTop 10 most probable bitstrings:")
sorted_idx = np.argsort(probs)[::-1]
for rank, idx in enumerate(sorted_idx[:10]):
    bitstr = format(idx, f"0{N_QUBITS}b")
    cut_val = all_cuts[idx]
    print(f"  |{bitstr}> : prob={probs[idx]:.4f}, cut={cut_val}")

In [ ]:
# --- Plot probability vs cut value ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: probability of each bitstring, coloured by cut value
colors_cut = plt.cm.RdYlBu_r(np.array(all_cuts) / max(all_cuts))
axes[0].bar(range(DIM), probs, color=colors_cut, width=1.0)
axes[0].set_xlabel("Bitstring index")
axes[0].set_ylabel("Probability")
axes[0].set_title("QAOA p=1 output state (colour = cut value)")

# Right: probability mass per cut value
unique_cuts = sorted(set(all_cuts))
prob_per_cut = [
    sum(probs[i] for i in range(DIM) if all_cuts[i] == c) for c in unique_cuts
]
bar_colors = ["#dc2626" if c == exact_maxcut else "#2563eb" for c in unique_cuts]
axes[1].bar(unique_cuts, prob_per_cut, color=bar_colors, edgecolor="black")
axes[1].set_xlabel("Cut value")
axes[1].set_ylabel("Total probability")
axes[1].set_title(f"Probability mass per cut value (red = MaxCut = {exact_maxcut})")

plt.tight_layout()
plt.show()

# Probability of sampling the optimal cut
prob_optimal = sum(probs[i] for i in range(DIM) if all_cuts[i] == exact_maxcut)
print(f"Probability of sampling a MaxCut solution: {prob_optimal:.4f}")

## 9. Deeper QAOA (p=2)

Adding a second QAOA layer doubles the variational freedom:

$$|\gamma_1, \beta_1, \gamma_2, \beta_2\rangle = e^{-i\beta_2 B} e^{-i\gamma_2 C} e^{-i\beta_1 B} e^{-i\gamma_1 C} |+\rangle^{\otimes n}$$

This gives four `expv` calls (vs two for p=1). uniqx may now split
these across multiple hardware backends if beneficial.

Since a full 4D sweep is expensive, we initialise layer 1 at the p=1 optimum
$(\gamma^*, \beta^*)$ and sweep only the layer-2 angles $(\gamma_2, \beta_2)$.
This is a common heuristic (the INTERP strategy).

In [ ]:
@tracing.to_module(name="qaoa_p2")
def qaoa_p2(C_ham, B_ham, psi, gamma1, beta1, gamma2, beta2):
    """Two-layer QAOA: (cost -> mixer) x 2 -> measure cost."""
    # Layer 1
    psi = ops.expv(C_ham, psi, gamma1, hermitian=True)
    psi = ops.expv(B_ham, psi, beta1, hermitian=True)
    # Layer 2
    psi = ops.expv(C_ham, psi, gamma2, hermitian=True)
    psi = ops.expv(B_ham, psi, beta2, hermitian=True)
    # Measure
    cost = ops.expect(C_ham, psi)
    return psi, cost


# Trace and preflight
mod_p2_test = qaoa_p2(C_ham, B_ham, psi_plus, 0.5, 0.5, 0.5, 0.5)
opts_p2 = preflight(mod_p2_test, client=client)

n_ops_p2 = len(mod_p2_test.functions[0].ops)
print(f"QAOA p=2 module: {n_ops_p2} ops")
print(
    f"  (vs p=1: {n_ops} ops — {n_ops_p2 - n_ops} additional ops for the second layer)"
)

print("\nPreflight comparison:")
print(f"  {'Depth':<8} {'Label':<24} {'Time':>10} {'Cost ($)':>12}")
print(f"  {'-' * 8} {'-' * 24} {'-' * 10} {'-' * 12}")
for opt in options:
    flag = " *" if opt.get("recommended") else ""
    print(
        f"  {'p=1':<8} {opt['label']:<24} {opt['total_time']:>10.2f}"
        f"  ${opt['total_cost_usd']:>10.6f}{flag}"
    )
for opt in opts_p2:
    flag = " *" if opt.get("recommended") else ""
    print(
        f"  {'p=2':<8} {opt['label']:<24} {opt['total_time']:>10.2f}"
        f"  ${opt['total_cost_usd']:>10.6f}{flag}"
    )

In [ ]:
# --- Local sweep around the p=1 optimum ---
#
# Initialise layer 1 at (gamma*, beta*) from p=1 and sweep layer 2.

N_G2 = 4
N_B2 = 4

gamma2_vals = np.linspace(0.05, 2 * np.pi, N_G2)
beta2_vals = np.linspace(0.05, np.pi, N_B2)

cost_p2 = np.zeros((N_G2, N_B2))

SELECTED_P2 = opts_p2.recommended["_idx"] if opts_p2.recommended else 0

print(f"Sweeping p=2 layer-2 angles: {N_G2} x {N_B2} = {N_G2 * N_B2} points")
print(f"Layer 1 fixed at: gamma1={best_gamma_p1:.4f}, beta1={best_beta_p1:.4f}")
t0_p2 = time.monotonic()

for ig, g2 in enumerate(gamma2_vals):
    for ib, b2 in enumerate(beta2_vals):
        mod = qaoa_p2(
            C_ham, B_ham, psi_plus, best_gamma_p1, best_beta_p1, float(g2), float(b2)
        )

        jid = submit(
            mod,
            client=client,
            runtime_inputs=[
                fmt_mat(C_ham, DIM, DIM),
                fmt_mat(B_ham, DIM, DIM),
                fmt_vec(psi_plus, DIM),
                fmt_scalar(best_gamma_p1),
                fmt_scalar(best_beta_p1),
                fmt_scalar(float(g2)),
                fmt_scalar(float(b2)),
            ],
            preflight_job_id=opts_p2.job_id,
            option_idx=SELECTED_P2,
        )
        res = get(jid, client=client)

        payload = res.get("payload", b"")
        if isinstance(payload, str):
            payload = payload.encode()
        out = parse_result(payload, ["psi_out", "cost"])
        cost_p2[ig, ib] = out["cost"][2][0]

    elapsed = time.monotonic() - t0_p2
    print(f"  gamma2 {ig + 1}/{N_G2} done ({elapsed:.1f}s)")

t_p2 = time.monotonic() - t0_p2

best_p2_idx = np.unravel_index(cost_p2.argmax(), cost_p2.shape)
best_gamma2 = float(gamma2_vals[best_p2_idx[0]])
best_beta2 = float(beta2_vals[best_p2_idx[1]])
best_cost_p2 = cost_p2[best_p2_idx]

print(f"\np=2 sweep done in {t_p2:.1f}s")
print("Best p=2 parameters:")
print(f"  gamma1={best_gamma_p1:.4f}, beta1={best_beta_p1:.4f} (from p=1)")
print(f"  gamma2={best_gamma2:.4f}, beta2={best_beta2:.4f}")
print(f"  <C> = {best_cost_p2:.4f}")
print(f"  Approximation ratio: {best_cost_p2 / exact_maxcut:.4f}")

In [ ]:
# --- Plot p=1 vs p=2 landscapes side by side ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: p=1 landscape
G1, B1 = np.meshgrid(gamma_vals, beta_vals, indexing="ij")
im1 = axes[0].pcolormesh(G1, B1, cost_landscape, cmap="RdYlBu_r", shading="auto")
axes[0].plot(best_gamma_p1, best_beta_p1, "k*", markersize=18)
axes[0].set_xlabel(r"$\gamma$", fontsize=12)
axes[0].set_ylabel(r"$\beta$", fontsize=12)
axes[0].set_title(f"p=1: max $\\langle C \\rangle$ = {best_cost_p1:.3f}")
fig.colorbar(im1, ax=axes[0])

# Right: p=2 landscape (layer 2 angles, layer 1 fixed)
G2, B2 = np.meshgrid(gamma2_vals, beta2_vals, indexing="ij")
im2 = axes[1].pcolormesh(G2, B2, cost_p2, cmap="RdYlBu_r", shading="auto")
axes[1].plot(best_gamma2, best_beta2, "k*", markersize=18)
axes[1].set_xlabel(r"$\gamma_2$", fontsize=12)
axes[1].set_ylabel(r"$\beta_2$", fontsize=12)
axes[1].set_title(f"p=2 (layer 2): max $\\langle C \\rangle$ = {best_cost_p2:.3f}")
fig.colorbar(im2, ax=axes[1])

fig.suptitle(
    f"QAOA Energy Landscapes ({N_QUBITS}-vertex MaxCut, exact = {exact_maxcut})",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

## 10. Results Summary

Compare QAOA at p=1 and p=2 against the exact MaxCut solution,
plus hardware comparison from preflight.

In [ ]:
print("=" * 72)
print(f"  QAOA MaxCut Results: {N_QUBITS} vertices, {len(edges)} edges")
print("=" * 72)
print(f"\n  Exact MaxCut:  {exact_maxcut} edges")
print(f"  Optimal partitions: {len(exact_partitions)}")
for bits in exact_partitions[:4]:
    bitstr = format(bits, f"0{N_QUBITS}b")
    print(f"    |{bitstr}>")

print(f"\n  {'Method':<20} {'<C>':>8} {'Approx ratio':>14} {'Parameters':>12}")
print(f"  {'-' * 20} {'-' * 8} {'-' * 14} {'-' * 12}")
print(f"  {'Random guess':<20} {len(edges) / 2:>8.3f} {0.5:>14.4f} {'0':>12}")
print(
    f"  {'QAOA p=1':<20} {best_cost_p1:>8.3f} {best_cost_p1 / exact_maxcut:>14.4f} {'2':>12}"
)
print(
    f"  {'QAOA p=2':<20} {best_cost_p2:>8.3f} {best_cost_p2 / exact_maxcut:>14.4f} {'4':>12}"
)
print(f"  {'Exact MaxCut':<20} {exact_maxcut:>8.3f} {1.0:>14.4f} {'brute force':>12}")

print("\n  Hardware comparison (from preflight):")
print(f"  {'Depth':<8} {'Hardware':<20} {'Time':>10} {'Cost ($)':>12} {'Error':>8}")
print(f"  {'-' * 8} {'-' * 20} {'-' * 10} {'-' * 12} {'-' * 8}")
for opt in options:
    flag = " *" if opt.get("recommended") else ""
    print(
        f"  {'p=1':<8} {opt['label']:<20} {opt['total_time']:>10.2f}"
        f"  ${opt['total_cost_usd']:>10.6f}"
        f"  {opt['max_error_rate']:>8.4f}{flag}"
    )
for opt in opts_p2:
    flag = " *" if opt.get("recommended") else ""
    print(
        f"  {'p=2':<8} {opt['label']:<20} {opt['total_time']:>10.2f}"
        f"  ${opt['total_cost_usd']:>10.6f}"
        f"  {opt['max_error_rate']:>8.4f}{flag}"
    )

print(f"\n  Total wall time: p=1 sweep={t_sweep:.1f}s, p=2 sweep={t_p2:.1f}s")

## Summary

| Aspect | Detail |
|:-------|:-------|
| **Problem** | MaxCut on a 5-vertex, 6-edge graph (exact MaxCut = 5) |
| **Algorithm** | QAOA at depth p=1 (2 params) and p=2 (4 params) |
| **Primitives** | `expv` (cost & mixer unitaries), `expect` (cut measurement) |
| **Initial state** | Uniform superposition $\|+\rangle^{\otimes 5}$ |
| **uniqx routing** | Automatic CPU/GPU/QPU selection via preflight cost model |

**Key takeaways:**

1. **QAOA improves with depth**: The approximation ratio increases from p=1 to p=2, concentrating more probability mass on optimal and near-optimal cuts.
2. **Warm starting across layers**: Fixing layer-1 angles at the p=1 optimum and sweeping only layer-2 angles is an effective heuristic (INTERP strategy) that avoids a full 4D search.
3. **Smooth landscape**: The p=1 energy landscape has a clear global structure with few local optima, making gradient-free optimisation straightforward for small instances.
4. **Hardware flexibility**: The same QAOA code runs on CPU (small graphs), GPU (medium), or QPU (large) — uniqx selects the best backend based on graph size, circuit depth, and available hardware.
5. **Scaling**: For larger graphs, the $2^n$ Hilbert space makes classical simulation intractable, but QAOA on a QPU requires only $O(p \cdot |E|)$ gates per evaluation — a natural fit for near-term quantum devices.